# Ensembles

In [1]:
import pandas as pd
from sklearn.ensemble import BaggingRegressor, BaggingClassifier, GradientBoostingRegressor, GradientBoostingClassifier, StackingRegressor, StackingClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, confusion_matrix, classification_report, make_scorer
from math import sqrt

In [2]:
clsf_data = pd.read_csv('../data/processed_smoke_detector.csv')
X_clsf = clsf_data.drop(['Fire Alarm'], axis=1)
y_clsf = clsf_data['Fire Alarm']
X_clsf_train, X_clsf_test, y_clsf_train, y_clsf_test = train_test_split(X_clsf, y_clsf, test_size=0.2)

In [3]:
regr_data = pd.read_csv('../data/processed_trip_duration.csv')
X_regr = regr_data.drop(['trip_duration'], axis=1)
y_regr = regr_data['trip_duration']
X_regr_train, X_regr_test, y_regr_train, y_regr_test = train_test_split(X_regr, y_regr, test_size=0.2)

## BaggingRegressor


In [4]:
bagging_regr = BaggingRegressor(
    estimator=DecisionTreeRegressor(
        criterion='squared_error',
        splitter= 'best',
        max_depth=15,
        min_samples_split=6,
        min_samples_leaf=7
    ),
    n_estimators=10,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

In [5]:
bagging_regr.fit(X_regr_train, y_regr_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=15,
                                                 min_samples_leaf=7,
                                                 min_samples_split=6),
                 max_samples=0.8, n_jobs=-1, random_state=42)

In [6]:
y_regr_pred = bagging_regr.predict(X_regr_test)

In [7]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 195.00464162103896
MSE: 75827.0414306921
RMSE: 275.36710302919647
MAPE: 0.573225284507673
R^2: 0.67


## BaggingClassifier


In [8]:
bagging_clsf = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        **{'criterion': 'entropy', 'splitter': 'best', 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'sqrt'}
    ),
    n_estimators=50,
    max_samples=0.8,
    random_state=42
)

In [9]:
bagging_clsf.fit(X_clsf_train, y_clsf_train)

BaggingClassifier(estimator=DecisionTreeClassifier(criterion='entropy',
                                                   max_depth=12,
                                                   max_features='sqrt',
                                                   min_samples_leaf=3),
                  max_samples=0.8, n_estimators=50, random_state=42)

In [10]:
y_clsf_pred = bagging_clsf.predict(X_clsf_test)

In [11]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1717    2]
 [   0 6531]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1719
           1       1.00      1.00      1.00      6531

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



## GradientBoostingRegressor


In [ ]:
param_grid = {
    'loss': ['squared_error', 'huber'],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'criterion': ['friedman_mse', 'squared_error'],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 3],
    'max_depth': [3, 5],
    'max_features': [None, 'sqrt'],
    'alpha': [0.85, 0.9]
}

fixed_params = {
    'n_estimators': 100,
    'random_state': 42,
    'min_weight_fraction_leaf': 0,
    'min_impurity_decrease': 0,
    'init': None
}

In [13]:
gb = GradientBoostingRegressor(**fixed_params)

grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring=make_scorer(mean_squared_error, greater_is_better=False),
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [14]:
grid_search.fit(X_regr_train, y_regr_train)

Fitting 5 folds for each of 1024 candidates, totalling 5120 fits


KeyboardInterrupt: 

In [15]:
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший MSE: {-grid_search.best_score_:.4f}")

AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

In [16]:
gb_regr = GradientBoostingRegressor(
    n_estimators=20,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [17]:
gb_regr.fit(X_regr_train, y_regr_train)

GradientBoostingRegressor(n_estimators=20, random_state=42)

In [18]:
y_regr_pred = gb_regr.predict(X_regr_test)

In [19]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 233.52150016987494
MSE: 100295.11261117655
RMSE: 316.6940362734615
MAPE: 0.657476537515114
R^2: 0.57


## GradientBoostingClassifier


In [20]:
gb_clsf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [21]:
gb_clsf.fit(X_clsf_train, y_clsf_train)

GradientBoostingClassifier(random_state=42)

In [22]:
y_clsf_pred = gb_clsf.predict(X_clsf_test)

In [23]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1717    2]
 [   0 6531]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1719
           1       1.00      1.00      1.00      6531

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



## StackingRegressor


In [24]:
estimators = [
    ('dt', DecisionTreeRegressor(max_depth=4)),
    ('svr', SVR(kernel='rbf'))
]

In [25]:

stacking_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    n_jobs=-1
)

In [26]:
stacking_reg.fit(X_regr_train, y_regr_train)

KeyboardInterrupt: 

In [ ]:
y_pred = stacking_reg.predict(X_regr_test)

In [ ]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

## StackingClassifier

In [ ]:
estimators = [
    ('dt', DecisionTreeClassifier()),
    ('svc', KNN())
]

In [ ]:
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
    n_jobs=-1   
)

In [ ]:
stacking_clf.fit(X_clsf_train, y_clsf_train)

StackingClassifier(estimators=[('dt', DecisionTreeClassifier()),
                               ('svc', KNeighborsClassifier())],
                   final_estimator=LogisticRegression())

In [ ]:
y_clsf_pred = stacking_clf.predict(X_clsf_test)

In [ ]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1800    0]
 [   0 6450]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1800
           1       1.00      1.00      1.00      6450

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250

